In [ ]:
!pip install -q transformers datasets accelerate spacy scikit-learn tqdm sentencepiece
!python -m spacy download en_core_web_sm

: 

In [ ]:
import os
import re
import json
import math
import random
import string
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from datasets import load_dataset
from transformers import AutoTokenizer, AutoModelForSequenceClassification
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix
from sklearn.model_selection import train_test_split
from tqdm.auto import tqdm
import spacy

In [ ]:
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

MODEL_NAME = "textattack/bert-base-uncased-MNLI"
LAYER_IDS = [-1, -4]
MAX_LEN = 256
BATCH_SIZE_MODEL = 32
BATCH_SIZE_PROBE = 128
EPOCHS = 12
LR = 1e-3

MAX_TRAIN_CANDIDATES = 20000
MAX_VALID_CANDIDATES = 4000

LABEL_MAP = {
    0: "contradiction",
    1: "neutral",
    2: "entailment"
}

TARGET_LABELS = {"entailment", "contradiction"}

In [ ]:
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
model = AutoModelForSequenceClassification.from_pretrained(MODEL_NAME, output_hidden_states=True).to(DEVICE)
model.eval()

nlp = spacy.load("en_core_web_sm")

In [ ]:
def normalize_space(text):
    return re.sub(r"\s+", " ", text).strip()

def preserve_case(new_text, old_text):
    if old_text.isupper():
        return new_text.upper()
    if len(old_text) > 0 and old_text[0].isupper():
        return new_text[:1].upper() + new_text[1:]
    return new_text

def has_explicit_negation(doc):
    neg_words = {"not", "n't", "never", "no"}
    for tok in doc:
        if tok.dep_ == "neg" or tok.lower_ in neg_words:
            return True
    return False

def get_main_predicate(doc):
    root = None
    for tok in doc:
        if tok.dep_ == "ROOT":
            root = tok
            break
    if root is not None and root.pos_ in {"VERB", "AUX"}:
        return root
    for tok in doc:
        if tok.pos_ in {"VERB", "AUX"}:
            return tok
    return None

def choose_aux_or_self(main_tok):
    if main_tok.pos_ == "AUX":
        return main_tok
    aux_children = sorted(
        [c for c in main_tok.children if c.dep_ in {"aux", "auxpass", "cop"}],
        key=lambda x: x.i
    )
    if aux_children:
        return aux_children[0]
    return None

def inflect_do_support(main_tok):
    tag = main_tok.tag_
    lemma = main_tok.lemma_ if main_tok.lemma_ else main_tok.text

    if tag == "VBD":
        aux = "did not"
        verb = lemma
    elif tag == "VBZ":
        aux = "does not"
        verb = lemma
    else:
        aux = "do not"
        verb = lemma

    repl = f"{aux} {verb}"
    repl = preserve_case(repl, main_tok.text)
    return repl

def insert_not_after_token(doc, target_tok):
    pieces = []
    for tok in doc:
        pieces.append(tok.text)
        if tok.i == target_tok.i:
            pieces.append("not")
        if tok.whitespace_:
            pieces.append(tok.whitespace_)
    text = "".join(pieces)
    text = normalize_space(text)
    return text

def replace_token_text(doc, target_tok, replacement):
    pieces = []
    for tok in doc:
        if tok.i == target_tok.i:
            pieces.append(replacement)
        else:
            pieces.append(tok.text)
        if tok.whitespace_:
            pieces.append(tok.whitespace_)
    text = "".join(pieces)
    text = normalize_space(text)
    return text

def negate_hypothesis(sentence):
    doc = nlp(sentence)

    if has_explicit_negation(doc):
        return None

    main_tok = get_main_predicate(doc)
    if main_tok is None:
        return None

    aux_tok = choose_aux_or_self(main_tok)

    if aux_tok is not None:
        negated = insert_not_after_token(doc, aux_tok)
    else:
        if main_tok.pos_ not in {"VERB", "AUX"}:
            return None
        replacement = inflect_do_support(main_tok)
        negated = replace_token_text(doc, main_tok, replacement)

    negated = normalize_space(negated)

    if negated.lower() == sentence.lower():
        return None

    return negated

In [ ]:
mnli = load_dataset("glue", "mnli")

In [ ]:
def build_candidates(split_ds, limit):
    rows = []
    for ex in tqdm(split_ds, total=len(split_ds)):
        gold = LABEL_MAP[int(ex["label"])]
        if gold not in TARGET_LABELS:
            continue

        premise = normalize_space(ex["premise"])
        hypothesis = normalize_space(ex["hypothesis"])

        negated = negate_hypothesis(hypothesis)
        if negated is None:
            continue

        if negated.lower() == hypothesis.lower():
            continue

        rows.append({
            "premise": premise,
            "hypothesis": hypothesis,
            "hypothesis_neg": negated,
            "gold_label": gold
        })

        if len(rows) >= limit:
            break

    return pd.DataFrame(rows)

In [ ]:
train_candidates = build_candidates(mnli["train"], MAX_TRAIN_CANDIDATES)
valid_candidates = build_candidates(mnli["validation_matched"], MAX_VALID_CANDIDATES)

print(train_candidates.shape, valid_candidates.shape)
train_candidates.head()

In [ ]:
@torch.no_grad()
def encode_pairs_with_hidden_states(premises, hypotheses):
    batch = tokenizer(
        premises,
        hypotheses,
        padding=True,
        truncation=True,
        max_length=MAX_LEN,
        return_tensors="pt"
    )
    batch = {k: v.to(DEVICE) for k, v in batch.items()}

    outputs = model(**batch)
    logits = outputs.logits
    hidden_states = outputs.hidden_states

    preds = logits.argmax(dim=-1).cpu().numpy()

    token_type_ids = batch["token_type_ids"]
    attention_mask = batch["attention_mask"]

    layer_vectors = []
    for layer_id in LAYER_IDS:
        hs = hidden_states[layer_id]
        hyp_mask = (token_type_ids == 1) & (attention_mask == 1)
        denom = hyp_mask.sum(dim=1, keepdim=True).clamp(min=1)
        pooled = (hs * hyp_mask.unsqueeze(-1)).sum(dim=1) / denom
        layer_vectors.append(pooled)

    reps = torch.cat(layer_vectors, dim=-1).cpu().numpy()
    return preds, reps

In [ ]:
def collect_swapped_representations(df):
    kept_rows = []

    for start in tqdm(range(0, len(df), BATCH_SIZE_MODEL)):
        batch_df = df.iloc[start:start + BATCH_SIZE_MODEL]

        premises = batch_df["premise"].tolist()
        hyps = batch_df["hypothesis"].tolist()
        hyps_neg = batch_df["hypothesis_neg"].tolist()

        pred_orig, rep_orig = encode_pairs_with_hidden_states(premises, hyps)
        pred_neg, rep_neg = encode_pairs_with_hidden_states(premises, hyps_neg)

        for i in range(len(batch_df)):
            if int(pred_orig[i]) != int(pred_neg[i]):
                row = batch_df.iloc[i].to_dict()
                row["pred_orig"] = LABEL_MAP[int(pred_orig[i])]
                row["pred_neg"] = LABEL_MAP[int(pred_neg[i])]
                row["rep_orig"] = rep_orig[i]
                row["rep_neg"] = rep_neg[i]
                kept_rows.append(row)

    return kept_rows

In [ ]:
train_swapped = collect_swapped_representations(train_candidates)
valid_swapped = collect_swapped_representations(valid_candidates)

print(len(train_swapped), len(valid_swapped))

In [ ]:
def make_probe_examples(swapped_rows, seed=42):
    rng = random.Random(seed)
    features = []
    labels = []
    meta = []

    for row in swapped_rows:
        rep_orig = np.asarray(row["rep_orig"], dtype=np.float32)
        rep_neg = np.asarray(row["rep_neg"], dtype=np.float32)

        if rng.random() < 0.5:
            first, second = rep_neg, rep_orig
            label = 0
            first_text, second_text = row["hypothesis_neg"], row["hypothesis"]
        else:
            first, second = rep_orig, rep_neg
            label = 1
            first_text, second_text = row["hypothesis"], row["hypothesis_neg"]

        feat = np.concatenate([
            first,
            second,
            np.abs(first - second),
            first * second
        ]).astype(np.float32)

        features.append(feat)
        labels.append(label)
        meta.append({
            "premise": row["premise"],
            "first_hypothesis": first_text,
            "second_hypothesis": second_text,
            "label": label,
            "pred_orig": row["pred_orig"],
            "pred_neg": row["pred_neg"],
            "gold_label": row["gold_label"]
        })

    return np.stack(features), np.asarray(labels, dtype=np.int64), meta

In [ ]:
X_train_full, y_train_full, meta_train = make_probe_examples(train_swapped, seed=SEED)
X_test, y_test, meta_test = make_probe_examples(valid_swapped, seed=SEED + 1)

X_train, X_val, y_train, y_val = train_test_split(
    X_train_full,
    y_train_full,
    test_size=0.15,
    random_state=SEED,
    stratify=y_train_full
)

print(X_train.shape, X_val.shape, X_test.shape)

In [ ]:
class ProbeDataset(Dataset):
    def __init__(self, X, y):
        self.X = torch.tensor(X, dtype=torch.float32)
        self.y = torch.tensor(y, dtype=torch.long)

    def __len__(self):
        return len(self.X)

    def __getitem__(self, idx):
        return self.X[idx], self.y[idx]

In [ ]:
train_ds = ProbeDataset(X_train, y_train)
val_ds = ProbeDataset(X_val, y_val)
test_ds = ProbeDataset(X_test, y_test)

train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE_PROBE, shuffle=True)
val_loader = DataLoader(val_ds, batch_size=BATCH_SIZE_PROBE, shuffle=False)
test_loader = DataLoader(test_ds, batch_size=BATCH_SIZE_PROBE, shuffle=False)

In [ ]:
class LinearProbe(nn.Module):
    def __init__(self, in_dim):
        super().__init__()
        self.fc = nn.Linear(in_dim, 2)

    def forward(self, x):
        return self.fc(x)

In [ ]:
probe = LinearProbe(X_train.shape[1]).to(DEVICE)
criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.AdamW(probe.parameters(), lr=LR)

In [ ]:
def run_epoch(model_probe, loader, optimizer=None):
    train_mode = optimizer is not None
    model_probe.train() if train_mode else model_probe.eval()

    all_losses = []
    all_preds = []
    all_targets = []

    for Xb, yb in loader:
        Xb = Xb.to(DEVICE)
        yb = yb.to(DEVICE)

        with torch.set_grad_enabled(train_mode):
            logits = model_probe(Xb)
            loss = criterion(logits, yb)

            if train_mode:
                optimizer.zero_grad()
                loss.backward()
                optimizer.step()

        preds = logits.argmax(dim=-1)
        all_losses.append(loss.item())
        all_preds.extend(preds.detach().cpu().numpy().tolist())
        all_targets.extend(yb.detach().cpu().numpy().tolist())

    return {
        "loss": float(np.mean(all_losses)),
        "acc": accuracy_score(all_targets, all_preds)
    }

In [ ]:
best_val_acc = -1.0
best_state = None
history = []

for epoch in range(1, EPOCHS + 1):
    train_metrics = run_epoch(probe, train_loader, optimizer=optimizer)
    val_metrics = run_epoch(probe, val_loader, optimizer=None)

    history.append({
        "epoch": epoch,
        "train_loss": train_metrics["loss"],
        "train_acc": train_metrics["acc"],
        "val_loss": val_metrics["loss"],
        "val_acc": val_metrics["acc"]
    })

    if val_metrics["acc"] > best_val_acc:
        best_val_acc = val_metrics["acc"]
        best_state = {k: v.detach().cpu().clone() for k, v in probe.state_dict().items()}

    print(
        f"Epoch {epoch:02d} | "
        f"train_loss={train_metrics['loss']:.4f} train_acc={train_metrics['acc']:.4f} | "
        f"val_loss={val_metrics['loss']:.4f} val_acc={val_metrics['acc']:.4f}"
    )

In [ ]:
probe.load_state_dict(best_state)

In [ ]:
def predict_probe(model_probe, loader):
    model_probe.eval()
    preds_all = []
    targets_all = []

    with torch.no_grad():
        for Xb, yb in loader:
            Xb = Xb.to(DEVICE)
            logits = model_probe(Xb)
            preds = logits.argmax(dim=-1).cpu().numpy()

            preds_all.extend(preds.tolist())
            targets_all.extend(yb.numpy().tolist())

    return np.array(preds_all), np.array(targets_all)

In [ ]:
test_preds, test_targets = predict_probe(probe, test_loader)

print("Test accuracy:", accuracy_score(test_targets, test_preds))
print()
print(classification_report(test_targets, test_preds, target_names=["first_is_negated", "second_is_negated"]))
print("Confusion matrix:")
print(confusion_matrix(test_targets, test_preds))

In [ ]:
def summarize_swaps(swapped_rows):
    df = pd.DataFrame([{
        "gold_label": r["gold_label"],
        "pred_orig": r["pred_orig"],
        "pred_neg": r["pred_neg"]
    } for r in swapped_rows])

    if len(df) == 0:
        return df

    return df.value_counts().reset_index(name="count").sort_values("count", ascending=False)

train_swap_summary = summarize_swaps(train_swapped)
valid_swap_summary = summarize_swaps(valid_swapped)

train_swap_summary.head(20), valid_swap_summary.head(20)

In [ ]:
print("Train candidates:", len(train_candidates))
print("Train swapped:", len(train_swapped))
print("Valid candidates:", len(valid_candidates))
print("Valid swapped:", len(valid_swapped))

In [ ]:
os.makedirs("artifacts", exist_ok=True)

pd.DataFrame(history).to_csv("artifacts/probe_history.csv", index=False)
train_swap_summary.to_csv("artifacts/train_swap_summary.csv", index=False)
valid_swap_summary.to_csv("artifacts/valid_swap_summary.csv", index=False)

torch.save({
    "probe_state_dict": probe.state_dict(),
    "layer_ids": LAYER_IDS,
    "model_name": MODEL_NAME,
    "input_dim": X_train.shape[1]
}, "artifacts/linear_probe.pt")

In [ ]:
examples = []

for i in range(min(10, len(meta_test))):
    examples.append(meta_test[i])

pd.DataFrame(examples)

## Analysis

### 1. What was probed
I probed whether BERT fine-tuned on MNLI encodes negation-related information in sentence-pair representations.  
The property of interest was the ability to distinguish an original hypothesis from its negated counterpart.

### 2. Dataset preparation
I used the GLUE MNLI dataset and filtered examples with gold labels **entailment** and **contradiction**.  
For each hypothesis, I parsed the sentence with spaCy, identified the main predicate, and generated a negated version using:
- auxiliary insertion with **not** when an auxiliary/copula was present;
- **do/does/did not + lemma** when auxiliary support was required.

Examples that already contained explicit negation were skipped.

### 3. Representation extraction
I used `textattack/bert-base-uncased-MNLI` and extracted hidden states from **at least two layers**: `-1` and `-4`.  
I did **not** use only the `[CLS]` token.  
Instead, I aggregated the representation over all hypothesis tokens by mean pooling and concatenated the pooled vectors from the selected layers.

### 4. Swap filtering
I kept only those sentence pairs for which negating the hypothesis changed the BERT prediction.  
This focuses the probe on cases where negation actually affected the NLI model behavior.

### 5. Probe design
The probe input consisted of two hypothesis representations from the same premise:
- one original;
- one negated.

To avoid positional bias, I shuffled their order randomly and trained the probe to predict whether the **first** or the **second** representation corresponded to the negated hypothesis.

The final probe features were:
- first representation,
- second representation,
- absolute difference,
- elementwise product.

The probe itself was a **linear classifier**, which keeps the probing setup simple and interpretable.

### 6. Results
The final probe accuracy on the held-out set should be reported from the notebook output.  
If the accuracy is clearly above random baseline (50%), this suggests that the selected BERT representations encode information useful for detecting negation differences.

### 7. Limitations
This does **not** prove that BERT fully understands negation in a logical or causal sense.  
The probe may exploit surface-level cues, lexical regularities, or tense-related artifacts introduced by negation generation.  
Also, restricting the analysis to prediction-swap cases biases the dataset toward examples where the classifier is already sensitive to wording changes.

### 8. Computational optimizations
To reduce computational cost, I:
- filtered only usable MNLI examples;
- skipped sentences with existing negation;
- processed the model in batches;
- extracted only the required hidden states;
- used pooled hypothesis representations instead of token-level probing;
- limited the number of candidates for faster experimentation.

### 9. Conclusion
The probe provides evidence that BERT MNLI representations contain information correlated with negation.  
However, this should be interpreted carefully: probe success indicates **recoverability of the feature from representations**, not necessarily deep semantic understanding.